In [0]:
from datetime import datetime, timezone
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

In [0]:
# ===============
# Workflow Types
# ===============

AgentName = Literal[
    "coordinator_agent",
    "sql_agent",
    "prediction_agent",
    "vector_search_agent",
    "retention_agent",
    "final_response_agent",
]

AgentStatus = Literal[
    "pending",
    "running",
    "success",
    "failed",
    "skipped",
]

RequestType = Literal[
    "sql_analytics",
    "prediction",
    "vector_search",
    "retention",
    "combined",
    "unsupported",
]

# ===============
# Business Types
# ===============

SupportCategory = Literal[
    "Billing",
    "Technical",
    "Service",
    "Account",
    "General",
]

RetentionAction = Literal[
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action",
]

In [0]:
# 1 : Coordinator task schema

class AgentTask(BaseModel):
    """
    Represents one task in the Coordinator Agent's execution plan.
    """

    task_id: str

    agent_name: AgentName

    task_description: str

    depends_on: List[AgentName] = Field(
        default_factory=list
    )

In [0]:
from pydantic import BaseModel, ConfigDict
from typing import Optional


class BaseAgentResult(BaseModel):
    """
    Common fields returned by every agent.
    """

    model_config = ConfigDict(
        extra="forbid"
    )

    agent_name: AgentName

    status: AgentStatus

    message: str

    task_description: Optional[str] = None

    error: Optional[str] = None

In [0]:
# 3 : Coordinator result schema

class CoordinatorResult(BaseAgentResult):
    """
    Validated result returned by the Coordinator Agent.
    """

    agent_name: Literal["coordinator_agent"] = "coordinator_agent"  

    request_type: RequestType

    reasoning: str

    execution_plan: List[AgentTask] = Field(
        default_factory=list
    )

In [0]:
# 4 : SQL Agent result schema

class SQLAgentResult(BaseAgentResult):
    """
    Validated result returned by the SQL Agent.
    """

    agent_name: Literal["sql_agent"] = "sql_agent"

    sql_action: Optional[str] = None

    sql_result: Any = None

    row_count: Optional[int] = None

# The sql_result field uses Any because the SQL tool may return: Spark Row objects, Lists of rows, Dictionaries, Numeric aggregates, Tabular results

In [0]:
# 5 : Prediction Agent result schema

ChurnPredictionCategory = Literal[
    "Churn",
    "No Churn",
]


class PredictionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Prediction Agent.
    """

    agent_name: Literal["prediction_agent"] = (
        "prediction_agent"
    )

    customer_id: Optional[str] = None

    predicted_category: Optional[
        ChurnPredictionCategory
    ] = None

    confidence: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

    model_name: Optional[str] = None

    raw_prediction: Any = None
# The raw_prediction field preserves the original Prediction Tool output when needed.

In [0]:
from typing import List, Optional
from pydantic import BaseModel, Field


class VectorSearchItem(BaseModel):
    """
    One customer-note result returned by semantic search.
    """

    customer_id: str

    note: str

    similarity_score: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

In [0]:
# 6 : Vector Search Agent result schema

class VectorSearchAgentResult(BaseAgentResult):
    """
    Validated output returned by the Vector Search Agent.
    """

    task_id: Optional[str] = None

    query: Optional[str] = None

    results: List[VectorSearchItem] = Field(
        default_factory=list
    )

In [0]:
# 7 : Retention Agent result schema

class RetentionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Retention Agent.
    """

    agent_name: Literal["retention_agent"] = (
        "retention_agent"
    )

    recommended_action: Optional[RetentionAction] = None

    recommendation: Optional[str] = None

    prediction_context: Optional[Dict[str, Any]] = None

    similar_ticket_context: List[Dict[str, Any]] = Field(
        default_factory=list
    )

In [0]:
# 8 : Final Response Agent result schema

class FinalResponseAgentResult(BaseAgentResult):
    """
    Validated result returned by the Final Response Agent.
    """

    agent_name: Literal["final_response_agent"] = (
        "final_response_agent"
    )

    final_response: Optional[str] = None

In [0]:
class AgentErrorRecord(BaseModel):
    """
    Represents one agent or workflow error.
    """

    agent_name: AgentName

    error_code: str

    error_message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(
            timezone.utc
        )
    )

In [0]:
# Execution-history schema: An ExecutionRecord represents one event in the workflow execution history.

class ExecutionRecord(BaseModel):
    """
    Represents one agent execution event in the multi-agent workflow.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

In [0]:
def validate_shared_schemas() -> None:
    """
    Run basic validation checks for shared schemas.
    Call manually only when testing this notebook.
    """

    example_task = AgentTask(
        task_id="task_1",
        agent_name="prediction_agent",
        task_description=(
            "Predict the support-ticket category."
        ),
    )

    example_prediction_result = PredictionAgentResult(
        status="success",
        message="Prediction completed successfully.",
        predicted_category="Technical",
        confidence=0.91,
    )

    example_execution_record = ExecutionRecord(
        agent_name="prediction_agent",
        status="success",
        message="Prediction task completed successfully.",
    )

    assert example_task.agent_name == "prediction_agent"
    assert (
        example_prediction_result.predicted_category
        == "Technical"
    )
    assert example_execution_record.status == "success"

    print("All shared-schema validation checks passed.")